# RLVR Run Dashboard

Use this after a smoke run or training run to inspect the learning signal. This is intentionally a notebook first: the views will evolve while we learn what matters.

Expected run command:

```bash
uv run python -m posttraining.rlvr.countdown_train max_steps=3
```

Then point `RUN_DIR` below at the run output directory.

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

RUN_DIR = Path("../outputs/rlvr_countdown_3num").resolve()
print(RUN_DIR)
print("exists:", RUN_DIR.exists())

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    for line in path.read_text().splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def metric_series(records: list[dict], key: str) -> tuple[list[int], list[float]]:
    xs, ys = [], []
    for row in records:
        if key in row and row[key] is not None:
            xs.append(int(row.get("step", len(xs))))
            ys.append(float(row[key]))
    return xs, ys


metrics = read_jsonl(RUN_DIR / "metrics.jsonl")
print("metrics rows:", len(metrics))
if metrics:
    print("sample keys:")
    print(sorted(metrics[-1].keys())[:80])
else:
    print("No metrics yet. Run a smoke experiment first.")

## Scalar learning curves

These are the first curves to watch. Reward alone is not enough; separate format, correctness, entropy, and mixed-group rate.

In [ ]:
curve_keys = [
    "env/all/reward/total",
    "env/all/correct",
    "env/all/format",
    "env/all/by_group/frac_mixed",
    "env/all/by_group/frac_all_bad",
    "env/all/by_group/frac_all_good",
    "optim/entropy",
    "optim/kl_sample_train_v2",
]

fig, axes = plt.subplots(4, 2, figsize=(12, 12), constrained_layout=True)
for ax, key in zip(axes.ravel(), curve_keys, strict=False):
    xs, ys = metric_series(metrics, key)
    ax.plot(xs, ys, marker="o")
    ax.set_title(key)
    ax.set_xlabel("step")
    ax.grid(alpha=0.25)

for ax in axes.ravel()[len(curve_keys) :]:
    ax.axis("off")

plt.show()

## Rollout summaries

Rollout summaries are one JSON record per sampled trajectory. They are the best artifact for understanding why a step did or did not produce useful learning signal.

In [ ]:
def rollout_paths(run_dir: Path, base_name: str = "train") -> list[Path]:
    return sorted(run_dir.glob(f"iteration_*/{base_name}_rollout_summaries.jsonl"))


paths = rollout_paths(RUN_DIR)
print("rollout files:", len(paths))
for path in paths[:5]:
    print(path.relative_to(RUN_DIR))

rollouts = []
for path in paths:
    rollouts.extend(read_jsonl(path))

print("rollout records:", len(rollouts))
if rollouts:
    print(rollouts[0].keys())

In [ ]:
def first_step_metrics(record: dict) -> dict:
    steps = record.get("steps") or []
    if not steps:
        return {}
    return steps[0].get("metrics") or {}


if rollouts:
    totals = [float(row["total_reward"]) for row in rollouts]
    print("reward histogram:", Counter(totals))
    for name in ["format", "correct"]:
        values = [first_step_metrics(row).get(name) for row in rollouts]
        print(name, Counter(values))
else:
    print("No rollout summaries yet.")

## Group signal view

For GRPO-style intuition, a useful group has reward variation. All-bad and all-good groups produce little centered-advantage signal.

In [ ]:
groups = defaultdict(list)
for row in rollouts:
    groups[(row["iteration"], row["group_idx"])].append(row)

group_rows = []
for key, rows in groups.items():
    rewards = [float(row["total_reward"]) for row in rows]
    mean_reward = sum(rewards) / len(rewards)
    group_rows.append(
        {
            "iteration": key[0],
            "group_idx": key[1],
            "n": len(rows),
            "mean_reward": mean_reward,
            "min_reward": min(rewards),
            "max_reward": max(rewards),
            "mixed": min(rewards) != max(rewards),
        }
    )

print("groups:", len(group_rows))
print("mixed groups:", sum(row["mixed"] for row in group_rows))
group_rows[:10]

In [ ]:
if group_rows:
    by_iteration = defaultdict(list)
    for row in group_rows:
        by_iteration[row["iteration"]].append(row)

    xs = sorted(by_iteration)
    ys = [sum(row["mixed"] for row in by_iteration[x]) / len(by_iteration[x]) for x in xs]

    plt.figure(figsize=(8, 4))
    plt.plot(xs, ys, marker="o")
    plt.title("Fraction of mixed-reward groups")
    plt.xlabel("iteration")
    plt.ylabel("frac mixed")
    plt.ylim(-0.05, 1.05)
    plt.grid(alpha=0.25)
    plt.show()

## Advantage intuition for one group

Advantages are centered rewards within a group: `advantage = reward - group_mean_reward`.

In [ ]:
mixed_key = None
for key, rows in groups.items():
    rewards = [float(row["total_reward"]) for row in rows]
    if min(rewards) != max(rewards):
        mixed_key = key
        break

if mixed_key is None:
    print("No mixed group found yet. Try more steps or larger group_size.")
else:
    rows = groups[mixed_key]
    rewards = [float(row["total_reward"]) for row in rows]
    mean_reward = sum(rewards) / len(rewards)
    print("iteration/group:", mixed_key)
    print("mean reward:", round(mean_reward, 3))
    for row in rows:
        reward = float(row["total_reward"])
        advantage = reward - mean_reward
        metrics = first_step_metrics(row)
        print(
            {
                "traj": row["traj_idx"],
                "reward": reward,
                "advantage": round(advantage, 3),
                "format": metrics.get("format"),
                "correct": metrics.get("correct"),
                "ac_len": (row.get("steps") or [{}])[0].get("ac_len"),
            }
        )

## Where to inspect full text

The rollout summaries intentionally stay compact. For full prompt/response text, open the logtree HTML/JSON files in the iteration directories.

In [ ]:
logtrees = sorted(RUN_DIR.glob("iteration_*/*_logtree.json"))
htmls = sorted(RUN_DIR.glob("iteration_*/*.html"))

print("logtree json files:", len(logtrees))
for path in logtrees[:10]:
    print(path.relative_to(RUN_DIR))

print("\nhtml files:", len(htmls))
for path in htmls[:10]:
    print(path.relative_to(RUN_DIR))